<a href="https://colab.research.google.com/github/KayotPoziloy/avito-test/blob/master/avito_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Кандидатогенерация для услуг (Авито)

Нужно для каждого запроса вернуть до 50 объявлений из корпуса, метрика Recall@50.

Итоговое решение: BM25 по тексту объявления + приоритет объявлений из локации запроса.

In [1]:
!pip install -q PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 14.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import re
import time

import numpy as np
import pandas as pd
import Stemmer
from sklearn.feature_extraction.text import CountVectorizer

DATA = '/content/drive/MyDrive/dataset/'
K = 50

In [4]:
queries = pd.read_parquet(DATA + 'benchmark_queries.parquet')
items = pd.read_parquet(DATA + 'benchmark_items.parquet')
# описания в train самые тяжёлые, для EDA и валидации они не нужны целиком
train = pd.read_parquet(DATA + 'train.parquet',
                        columns=['search_query', 'search_location_id', 'search_infm_params_text',
                                 'search_category', 'item_id', 'item_title_raw',
                                 'item_infm_params_text', 'item_location_id',
                                 'item_microcat_id', 'item_rating_reviews_count'])

queries['search_infm_params_text'] = queries['search_infm_params_text'].fillna('')
train['search_infm_params_text'] = train['search_infm_params_text'].fillna('')

train.shape, queries.shape, items.shape

((497673, 10), (2452, 6), (189212, 14))

## EDA

In [5]:
train.head(3).T

,0,1,2
search_query,скупка телевизоров,автоподбор,баня на дровах
search_location_id,652430,640860,653240
search_infm_params_text,,Рейтинг пользователя 4 звезды и выше,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви..."
search_category,114,114,114
item_id,e8b685dffe1a408e,92e1b0446f827b59,624846856ce81d69
item_title_raw,Скупка б/у техники,Автоподбор Разовый осмотр автомобиля,"Баня на дровах ""Прованс"" на Цветочной"
item_infm_params_text,Вид услуги Место оказания услуг Первомайский п...,Вид услуги Место оказания услуг Нижний Новгоро...,"Вид услуги Красота, здоровье Место оказания ус..."
item_location_id,652430,640860,653240
item_microcat_id,2303374,2303374,86469
item_rating_reviews_count,91.0,462.0,NaN


In [6]:
# сколько объявлений из train есть в корпусе
train['item_id'].isin(items['item_id']).mean()

np.float64(0.06632869374066908)

Всего 6.6%, т.е. просто брать объявления, выбранные в train по такому же запросу, не получится.

In [7]:
# доля запросов бенчмарка, которые уже встречались в train
seen = set(train['search_query'].str.lower().str.strip())
queries['search_query'].str.lower().str.strip().isin(seen).mean()

np.float64(0.3699021207177814)

In [8]:
# совпадает ли локация запроса и выбранного объявления
(train['search_location_id'] == train['item_location_id']).mean()

np.float64(0.8310195650557696)

83% - локация очень сильный сигнал, надо её использовать.

In [9]:
queries['search_query'].str.split().str.len().describe()

,search_query
count,2452.000000
mean,3.197390
std,1.301298
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,10.000000


In [10]:
# фильтры поиска: похожи на параметры объявлений ("Вид услуги ...")
print((queries['search_infm_params_text'] != '').mean())
queries.loc[queries['search_infm_params_text'] != '', 'search_infm_params_text'].value_counts().head(10)

0.3694942903752039


,count
search_infm_params_text,
Вид услуги,60
"Вид услуги Оборудование, производство",53
Вид услуги Ремонт и отделка,48
"Вид услуги Праздники, мероприятия",40
"Вид услуги Обучение, курсы",39
"Вид услуги Красота, здоровье",36
Вид услуги Вывоз мусора и вторсырья,36
Вид услуги Грузоперевозки,31
"Тип услуги Услуги парикмахера Вид услуги Красота, здоровье",26


In [11]:
train['search_category'].value_counts().head()

,count
search_category,
114,497635
0,34
81,2
10,2


Запросы короткие (в среднем 3 слова), категория почти всегда одна и та же (114), её не используем.
Для начала пробую BM25 по тексту объявления.

## Токенизация

Приводим к нижнему регистру, режем на слова и стеммим (Snowball, PyStemmer). Стеммер вызываю только для новых слов, так быстрее.

In [12]:
stemmer = Stemmer.Stemmer('russian')
token_re = re.compile(r'[а-яёa-z0-9]+')
stem_cache = {}


def tokenize(texts):
    tokens = [token_re.findall(t.lower().replace('ё', 'е')) for t in texts]
    new_words = {w for ts in tokens for w in ts} - stem_cache.keys()
    if new_words:
        new_words = list(new_words)
        stem_cache.update(zip(new_words, stemmer.stemWords(new_words)))
    return [' '.join(stem_cache[w] for w in ts) for ts in tokens]


tokenize(['Ремонт холодильников на дому', 'ремонт холодильника'])

['ремонт холодильник на дом', 'ремонт холодильник']

## BM25

Готовую библиотеку не брал, BM25 считается через разреженные матрицы:
веса BM25 для каждой пары (документ, слово) считаются один раз, дальше скоры для пачки
запросов - это просто `Q @ W`.

Текст документа: заголовок два раза (он важнее) + параметры + первые 1000 символов описания.

Топ-50 собираю так:
1. n_local лучших по BM25 среди объявлений из той же локации
2. остальное - лучшие по всему корпусу (например, перевозки между городами)
3. если BM25 ничего не нашёл, добиваю популярными (по числу отзывов) объявлениями из локации

In [13]:
class BM25Index:
    def __init__(self, corpus, title_w=2, desc_chars=1000, k1=1.2, b=0.75):
        t0 = time.time()
        title = corpus['item_title_raw'].fillna('')
        params = corpus['item_infm_params_text'].fillna('')
        desc = corpus['item_description_raw'].fillna('').str[:desc_chars]
        docs = tokenize(((title + ' ') * title_w + params + ' ' + desc).tolist())

        self.vec = CountVectorizer(analyzer=str.split, dtype=np.float32)
        tf = self.vec.fit_transform(docs).tocsr()   # документы x слова

        n = tf.shape[0]
        doc_len = np.asarray(tf.sum(1)).ravel()
        df = np.bincount(tf.indices, minlength=tf.shape[1])
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

        # BM25 для каждого ненулевого элемента
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))
        denom = tf.data + k1 * (1 - b + b * doc_len[rows] / doc_len.mean())
        tf.data = tf.data * (k1 + 1) / denom * idf[tf.indices]
        self.W = tf.T.tocsr()                        # слова x документы

        self.ids = corpus['item_id'].to_numpy()
        self.pop = corpus['item_rating_reviews_count'].fillna(0).to_numpy()
        loc = corpus['item_location_id'].to_numpy()
        self.loc_index = pd.Series(loc).groupby(loc).indices   # локация -> индексы документов
        print(f'{n} docs, {tf.shape[1]} words, {time.time() - t0:.0f}s')

    def query_matrix(self, texts):
        q = self.vec.transform(tokenize(texts))
        q.data[:] = 1
        return q

    def search(self, qdf, n_local=40, use_filters=True, batch=256):
        texts = qdf['search_query'].fillna('')
        if use_filters:
            texts = texts + ' ' + qdf['search_infm_params_text']
        Q = self.query_matrix(texts.tolist())
        locs = qdf['search_location_id'].to_numpy()
        res = []
        for s in range(0, Q.shape[0], batch):
            scores = (Q[s:s + batch] @ self.W).toarray()
            for i, sc in enumerate(scores):
                res.append(self.top_k(sc, locs[s + i], n_local))
        return res

    def top_k(self, scores, loc, n_local):
        chosen = []
        local = self.loc_index.get(loc, np.array([], dtype=int))
        if len(local):
            top = local[np.argsort(-scores[local])[:n_local]]
            chosen += [j for j in top if scores[j] > 0]

        seen = set(chosen)
        m = min(200, len(scores) - 1)
        cand = np.argpartition(-scores, m)[:m]
        for j in cand[np.argsort(-scores[cand])]:
            if len(chosen) >= K or scores[j] <= 0:
                break
            if j not in seen:
                chosen.append(j)
                seen.add(j)

        if len(chosen) < K and len(local):
            for j in local[np.argsort(-self.pop[local])]:
                if len(chosen) >= K:
                    break
                if j not in seen:
                    chosen.append(j)
                    seen.add(j)
        return list(self.ids[chosen[:K]])

## Валидация

Группирую train по (запрос, локация, фильтры) - это аналог одного запроса из бенчмарка,
все выбранные объявления группы считаю правильными. Отложил 3000 групп.
Их объявлений в корпусе почти нет, поэтому добавляю их в корпус, иначе искать нечего.
Из-за этого валидация получается немного оптимистичнее реальности (0.83 против 0.75 на платформе),
но сравнивать варианты между собой по ней можно.

In [14]:
def recall_at_k(preds, positives, k=K):
    return np.mean([len(set(p[:k]) & pos) / len(pos) for p, pos in zip(preds, positives)])


train['q_norm'] = train['search_query'].fillna('').str.lower().str.strip()
groups = (train.groupby(['q_norm', 'search_location_id', 'search_infm_params_text'], sort=False)
               .agg(search_query=('search_query', 'first'),
                    positives=('item_id', lambda x: set(x)))
               .reset_index())

rng = np.random.default_rng(42)
val_q = groups.iloc[rng.choice(len(groups), 3000, replace=False)].reset_index(drop=True)
val_pos = val_q['positives'].tolist()

# описания подгружаю только для нужных объявлений
val_ids = set().union(*val_pos)
desc = (pd.read_parquet(DATA + 'train.parquet', columns=['item_id', 'item_description_raw'],
                        filters=[('item_id', 'in', list(val_ids))])
          .drop_duplicates('item_id').set_index('item_id')['item_description_raw'])
extra = train[train['item_id'].isin(val_ids)].drop_duplicates('item_id').copy()
extra['item_description_raw'] = extra['item_id'].map(desc)

val_corpus = pd.concat([items, extra[items.columns.intersection(extra.columns)]])
val_corpus = val_corpus.drop_duplicates('item_id').reset_index(drop=True)
len(val_corpus), np.mean([len(p) for p in val_pos])

(192925, np.float64(1.3246666666666667))

In [15]:
val_index = BM25Index(val_corpus)

for n_local in [0, 30, 40, 45, 50]:
    for use_filters in [False, True]:
        preds = val_index.search(val_q, n_local=n_local, use_filters=use_filters)
        print(n_local, use_filters, round(recall_at_k(preds, val_pos), 4))

192925 docs, 288257 words, 103s
0 False 0.3179
0 True 0.3202
30 False 0.8153
30 True 0.8289
40 False 0.8181
40 True 0.8335
45 False 0.8188
45 True 0.8326
50 False 0.8179
50 True 0.8312


Без локации (n_local=0) recall 0.32, с локацией сразу 0.82 - основной прирост отсюда.
Фильтры в тексте запроса дают ещё +1.5 п.п. Лучшее: n_local=40 с фильтрами (0.8335).

Ещё пробовал добавить эмбеддинги multilingual-e5-base и смешивать с BM25 через RRF - на валидации
было 0.8488 (+1.5 п.п.), но в Colab не хватало памяти держать всё сразу, в итоговую версию не вошло.

## Анализ ошибок

Смотрю на правильные объявления, которые не попали в топ-50 (при n_local=40 с фильтрами),
и делю промахи на типы:
* нет общих слов с запросом - BM25 такое найти не может в принципе (синонимы, другие формулировки)
* объявление из другой локации - попадает только в 10 "общих" мест
* слова совпали, локация та же, но выше оказались другие объявления

In [16]:
best = val_index.search(val_q, n_local=40, use_filters=True)
Q = val_index.query_matrix((val_q['search_query'].fillna('') + ' ' + val_q['search_infm_params_text']).tolist())
row_of = pd.Series(np.arange(len(val_corpus)), index=val_corpus['item_id'])
item_loc = val_corpus['item_location_id'].to_numpy()

rows = []
for i, (pred, pos) in enumerate(zip(best, val_pos)):
    missed = pos - set(pred)
    if not missed:
        continue
    scores = (Q[i] @ val_index.W).toarray().ravel()
    q_loc = val_q.loc[i, 'search_location_id']
    local = val_index.loc_index.get(q_loc, np.array([], dtype=int))
    for item in missed:
        j = row_of[item]
        if scores[j] == 0:
            kind = 'нет общих слов'
        elif item_loc[j] != q_loc:
            kind = 'другая локация'
        else:
            kind = 'проиграл другим'
        rows.append({'query': val_q.loc[i, 'search_query'], 'title': val_corpus.loc[j, 'item_title_raw'],
                     'kind': kind, 'rank_in_loc': int((scores[local] > scores[j]).sum()) + 1,
                     'loc_size': len(local)})
errors = pd.DataFrame(rows)

print('промахов:', len(errors), 'из', sum(len(p) for p in val_pos))
errors['kind'].value_counts(normalize=True).round(3)

промахов: 680 из 3974


,proportion
kind,
другая локация,0.632
проиграл другим,0.303
нет общих слов,0.065


In [17]:
# примеры: нет общих слов
errors[errors['kind'] == 'нет общих слов'][['query', 'title']].sample(frac=1, random_state=1).head(15)

,query,title
61,вибромассаж,Сферамассаж / Аппаратный массаж сфера
59,риелтор,Услуги риэлтора
503,чистка кондиционеров,Юрист
376,отметка гбо,Регистрация переоборудования тс
390,погрузка трактором,Услуги минипогрузчика
675,нашивки шевроны,"Вышивка на одежде с логотипом, форма для персо..."
343,копка траншей,Ручная перекопка огорода
615,самосвал,Вывоз грунта
562,пескоструйка,Аренда пескоструйного аппарата
546,повешать люстру,Услуги мастер на час


In [18]:
# примеры: другая локация
errors[errors['kind'] == 'другая локация'][['query', 'title']].sample(frac=1, random_state=1).head(10)

,query,title
377,отметка гбо,Переоборудование транспортных средств
344,скупка ноутбуков,Выкуп ноутбуков. Быстрая оценка и деньги сразу
281,кинолог,Дрессировка и обучение собак кинолог
454,сварка аргоном,"Сварочные работы по кузову автомобиля, аргон"
93,детский развивающий центр,Частный детский клуб
289,догруз,"Междугородние грузоперевозки, Домашние переезды"
525,установка опор лэп,"Установка, монтаж, демонтаж опор лэп в Химках"
7,ремонт мопедов,"Ремонт квадроциклов, мотоциклов, снегоходов"
404,аппаратный массаж,"Массаж ""Лепка тела"""
625,ремонт гбц ваз,Ремонт двигателя(двс).Ремонт гбц. Замена грм


In [19]:
# проиграл другим: на каком месте в своей локации и насколько локация большая
errors[errors['kind'] == 'проиграл другим'][['rank_in_loc', 'loc_size']].describe().round(0)

,rank_in_loc,loc_size
count,206.0,206.0
mean,237.0,8012.0
std,1005.0,7326.0
min,41.0,90.0
25%,56.0,3133.0
50%,83.0,3281.0
75%,132.0,12313.0
max,13652.0,19821.0


Что видно (680 промахов из 3974 правильных объявлений):

* **другая локация - 63%**. Текст часто совпадает хорошо ("кинолог" -> "Дрессировка и обучение собак кинолог",
  "установка опор лэп" -> "... опор лэп в Химках"), но объявление из другого location_id и не пролезает
  в 10 общих мест. Похоже, что часто это соседние населённые пункты (Химки / Москва).
* **проиграл другим - 30%**. Медиана места в своей локации 83 при отсечке 40, локации большие
  (медиана ~3.3 тыс. объявлений). Нужное объявление близко, но конкурентов слишком много.
* **нет общих слов - 6.5%**. Разное написание (риелтор / риэлтора), составные и жаргонные слова
  (пескоструйка, антикор, сферамассаж), синонимы (повешать люстру -> мастер на час). Есть и шум
  в разметке (чистка кондиционеров -> Юрист).

освобождаю память перед построением индекса для сабмита

In [20]:
del val_index, val_corpus, groups, extra, desc, train

## Сабмит

In [21]:
index = BM25Index(items)
preds = index.search(queries, n_local=40, use_filters=True)

answer = pd.DataFrame({'query_id': queries['query_id'],
                       'answer': [' '.join(p) for p in preds]})

# проверка формата
valid_ids = set(items['item_id'])
assert answer['query_id'].is_unique and set(answer['query_id']) == set(queries['query_id'])
assert all(0 < len(p) <= K and len(set(p)) == len(p) and set(p) <= valid_ids for p in preds)

answer.to_csv('answer.csv', index=False)
answer.head()

189212 docs, 284469 words, 87s


,query_id,answer
0,70DfDUpwjxB4lzFd,9515d1e1ecdac72c 14d535285758af89 a1f89a5af43b...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 422d3ffdd5bbf626 16ea26ff0aa0...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 3f89b8062dc85f1c d8fce513e4f0...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 67b2cd847669eb36 1e5924dc546b...
4,YgHcM9MVbxKnxD1e,1a62634394544781 53f9c8caeeb8111e 615c73ea4c3b...


## Описание решения

**Данные и признаки**

* Текст запроса + фильтры поиска (`search_infm_params_text`). Фильтры вида "Вид услуги ..." совпадают
  по словам с параметрами объявлений, на валидации это дало +1.5 п.п.
* Текст объявления: заголовок (с весом 2, он короткий и самый точный), параметры
  (`item_infm_params_text`) и первые 1000 символов описания.
* Локация: `search_location_id` и `item_location_id`. По train в 83% случаев выбранное объявление
  из той же локации, что и запрос, поэтому это главный признак: без него recall 0.32, с ним 0.83.
* Число отзывов - только чтобы добить список до 50, если по тексту мало совпадений.
* `search_category` не использовал (почти всегда 114). Объявления из train напрямую тоже не использовал:
  в корпусе их всего 6.6%.

**Алгоритм**

BM25 (k1=1.2, b=0.75), реализован сам через разреженные матрицы scipy, слова приводятся к основе
стеммером Snowball (PyStemmer). Из 50 мест 40 отдаются лучшим по BM25 объявлениям из локации запроса,
остальные - лучшим по всему корпусу, если не набралось 50 - популярные объявления локации.
Нейросетей и внешних API в итоговом решении нет.

Библиотеки: pandas, numpy, scipy, scikit-learn (CountVectorizer), PyStemmer.

**Как проверял качество**

Локальная валидация на train: 3000 случайных групп (запрос, локация, фильтры), правильные ответы -
выбранные в группе объявления, их добавил в корпус. По ней выбирал n_local и фильтры.
Валидация оптимистичнее платформы (0.8335 против 0.7477), потому что нужные объявления добавлены
в корпус искусственно, но порядок вариантов по ней адекватный.

**Ошибки и что с ними сделал**

* Объявление из другой локации (63% промахов). Поэтому локацию не сделал жёстким фильтром:
  10 мест из 50 остаются на весь корпус (n_local=40 лучше, чем 50). Дальше стоило бы учитывать
  близость по координатам (`item_latitude`/`item_longitude`) вместо точного совпадения location_id.
* Нужное объявление близко, но проигрывает конкурентам в большой локации (30%). Помогли вес заголовка
  и фильтры в запросе. Дальше - второй источник кандидатов с другими ошибками.
* Нет общих слов с запросом (6.5%). Стемминг закрывает часть форм слов. Пробовал эмбеддинги
  multilingual-e5-base + слияние с BM25 через Reciprocal Rank Fusion: на валидации 0.8488 (+1.5 п.п.),
  но в Colab не хватало памяти держать всё одновременно, в итоговую версию не вошло.
  Ещё может помочь поиск по символьным n-граммам (риелтор / риэлтор).
* Запросы, по которым BM25 почти ничего не нашёл, добиваю популярными объявлениями локации,
  чтобы не оставлять пустые места.